# Sensitivity to the observation period

Compare the full-period results with **30 May–9 June 2025**, for the same 37
sensors and three indicators: human presence ratio, bird presence ratio, and the
SPL summary used in the paper's location overview. Sensor 019 lacks records on
nine of these eleven dates and is excluded from this comparison only.

All queries use stored naive timestamps without further offsets.

Run from the repository root in `delft-sensors`. The database is read-only;
outputs are restricted to `out/observation_period_sensitivity/`. Full-period SED counts are
computed from the prepared source lake; full-period SPL summaries are reused from
`loudness_per_hour`. Common-period SED and SPL are freshly aggregated from the lake. Run the coverage script first.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, sys
import duckdb
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from IPython.display import display

from analysis_paths import connect_analysis, data_root, database_path, output_path, bundle_fingerprint
ROOT = Path.cwd()
OUT = output_path('out/observation_period_sensitivity')
OUT.mkdir(parents=True, exist_ok=True)
COVERAGE = output_path('out/temporal_coverage')
START, STOP = '2025-05-30', '2025-06-10'
CLASSES = ['human', 'bird']
checks, queries = {}, {}
started = datetime.now(timezone.utc).isoformat()

def check(name, ok):
    checks[name] = bool(ok)
    assert ok, name

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def query(name, sql, params=None):
    queries[name] = {'sql': sql, 'parameters': params or []}
    return db.execute(sql, params or []).df()

db = connect_analysis()
source_files = [data_root()/'manifest.json', COVERAGE/'audit_checks.json',
                COVERAGE/'calendar_inclusion.csv', COVERAGE/'daily_coverage.csv']
source_hashes = {str(p): sha(p) for p in source_files}
db_before = database_path().stat()
db_fingerprint = (db_before.st_size, db_before.st_mtime_ns)
print('Interpreter:', sys.executable)
print('Fixed comparison:', START, 'inclusive to', STOP, 'exclusive')

## 1. Selection, baselines and synthetic checks

An observed day has at least one retained record. Partial days remain included;
there is no new daily/hourly eligibility threshold. Coverage must agree with the
existing audit. Missing scores are excluded from the class-specific SED
denominator and explicitly counted; exactly 0.5 is not a detection.

The SPL indicator is **10 log10 of the median of 24 hour-of-day medians of
linear SPL**. It is neither a pooled median over all samples nor an equivalent
sound level. The final median weights the 24 clock hours equally; observations
within each hour's median retain their original sample weighting.

In [ ]:
calendar = pd.read_csv(COVERAGE/'calendar_inclusion.csv', parse_dates=['date'])
old_daily = pd.read_csv(COVERAGE/'daily_coverage.csv', parse_dates=['date'])
coverage_manifest = json.loads((COVERAGE/'audit_checks.json').read_text(encoding='utf-8'))
check('coverage uses this data bundle', coverage_manifest['data_manifest_sha256'] == bundle_fingerprint())
core = calendar[calendar.date.ge(START) & calendar.date.lt(STOP)]
eligible = core.groupby('sensor_id')[['spl_observed','sed_observed']].sum()
sensors = eligible.index[(eligible.spl_observed == 11) & (eligible.sed_observed == 11)].tolist()
check('exact 37-sensor cohort', len(sensors) == 37 and
      set(sensors) == set(calendar.sensor_id)-{'citizen-019'})
pd.DataFrame([{'sensor_id':'citizen-019','reason':'Only 2 of 11 common-window dates observed',
               'scope':'Excluded from both sides of sensitivity comparison only'}]).to_csv(
    OUT/'exclusions.csv', index=False)
fields = ['COUNT(*) AS total_rows', 'MIN(time) AS first_time', 'MAX(time) AS last_time']
for label in CLASSES:
    fields += [f'COUNT({label}) AS {label}_valid',
               f'COUNT(*) FILTER (WHERE {label} > 0.5) AS {label}_hits_1',
               f'100.0*AVG(({label} > 0.5)::DOUBLE) AS {label}_pct_1']
aud = query('full_period_SED_counts', 'SELECT sensor_id, '+', '.join(fields)+
    ' FROM sources GROUP BY sensor_id ORDER BY sensor_id').set_index('sensor_id')
views = dict(db.sql("SELECT view_name, sql FROM duckdb_views() WHERE view_name IN ('loudness','sources')").fetchall())
full_sed = []
for sid in sensors:
    saved = old_daily[(old_daily.sensor_id == sid) & (old_daily.dataset == 'sed_scores')]
    a = aud.loc[sid]
    check(sid+' full SED row count', saved.rows.sum() == a.total_rows)
    check(sid+' full SED timestamp extrema',
          pd.to_datetime(saved.first_time).min() == pd.Timestamp(a.first_time) and
          pd.to_datetime(saved.last_time).max() == pd.Timestamp(a.last_time))
    for label in CLASSES:
        valid, hits = int(a[label+'_valid']), int(a[label+'_hits_1'])
        check(sid+' '+label+' saved denominator', valid == a.total_rows and 0 <= hits <= valid)
        check(sid+' '+label+' saved arithmetic', np.isclose(100*hits/valid,a[label+'_pct_1'],atol=1e-10))
        full_sed.append(dict(sensor_id=sid,indicator=label,rows=int(a.total_rows),
            valid_scores=valid,null_scores=int(a.total_rows)-valid,hits=hits,value=100*hits/valid))
full_sed = pd.DataFrame(full_sed)
full_sed.to_csv(OUT/'full_period_sed_counts.csv', index=False)
full_hourly = query('full_period_SPL_baseline',
    "SELECT sensor_id, hour, median_spla FROM loudness_per_hour ORDER BY sensor_id, hour")
full_hourly = full_hourly[full_hourly.sensor_id.isin(sensors)].copy()
check('24 distinct valid SPL clock hours for every baseline sensor',
      len(full_hourly)==37*24 and
      full_hourly.groupby('sensor_id').hour.nunique().eq(24).all() and
      np.isfinite(full_hourly.median_spla).all() and full_hourly.median_spla.gt(0).all())
full_hourly.to_csv(OUT/'full_period_spl_hourly_medians.csv', index=False)

# These fixtures use the same operators and aggregation order as the analysis.
t = db.sql("""SELECT count(*) n, count(score) AS valid_count,
    count(*) FILTER (WHERE score > 0.5) hits
    FROM (VALUES (TIMESTAMP '2025-05-29 23:59:59',0.9),
                 (TIMESTAMP '2025-05-30 00:00:00',0.5),
                 (TIMESTAMP '2025-06-09 23:59:59',0.6),
                 (TIMESTAMP '2025-06-01 12:00:00',NULL),
                 (TIMESTAMP '2025-06-10 00:00:00',0.9)) t(time,score)
    WHERE time >= TIMESTAMP '2025-05-30' AND time < TIMESTAMP '2025-06-10'""").fetchone()
check('fixture: date boundaries, strict tie, NULL denominator', t == (3,2,1))
undefined = db.sql("SELECT 100.0*0/NULLIF(count(x),0) FROM (VALUES (NULL::DOUBLE)) t(x)").fetchone()[0]
check('fixture: undefined ratio is NULL', undefined is None)
f = db.sql("""WITH hourly AS (
    SELECT h, median(spl) m FROM (VALUES (0,1.0),(0,9.0),(1,100.0)) t(h,spl) GROUP BY h)
    SELECT 10*log10(median(m)) FROM hourly""").fetchone()[0]
check('fixture: two-stage linear SPL median', np.isclose(f,10*np.log10(52.5)))
check('fixture distinguishes pooled median', not np.isclose(f,10*np.log10(9)))
check('fixture distinguishes median in dB', not np.isclose(f,np.median(10*np.log10([5,100]))))
print('Baseline and synthetic checks passed; selected sensors:', len(sensors))

## 2. Fresh common-period SED and raw SPL

SED is aggregated jointly for human and bird. SPL is queried one sensor at a
time with date-partition and timestamp bounds. The minute mart is not used for
SPL values. Coverage measures count observed minute bins, not continuous uptime.

In [ ]:
selection_sql = ','.join("'"+s+"'" for s in sensors)
window_sql = f"date >= DATE '{START}' AND date < DATE '{STOP}' AND time >= TIMESTAMP '{START}' AND time < TIMESTAMP '{STOP}'"
class_sql = ',\n'.join(f"count({c}) AS {c}_valid, count(*) FILTER (WHERE {c} > 0.5) AS {c}_hits, count(*) FILTER (WHERE {c} IS NOT NULL AND (NOT isfinite({c}) OR {c}<0 OR {c}>1)) AS {c}_invalid" for c in CLASSES)
sed_hourly = query('common_SED_hourly', f"""
    SELECT sensor_id, cast(time AS DATE) AS day, date_trunc('hour',time) hour_start,
        count(*) AS rows, min(time) first_time, max(time) last_time,
        count(DISTINCT date_trunc('minute',time)) observed_minutes, {class_sql}
    FROM sources WHERE {window_sql} AND sensor_id IN ({selection_sql})
    GROUP BY 1,2,3 ORDER BY 1,2,3
""")
sed_hourly.to_csv(OUT/'common_sed_hourly.csv', index=False)
check('common SED class scores finite and within [0,1]',
      sed_hourly[[c+'_invalid' for c in CLASSES]].sum().sum() == 0)
spl_coverage_parts, spl_median_parts = [], []
for index,sid in enumerate(sensors,1):
    cov = query('common_SPL_coverage_'+sid, f"""
        SELECT sensor_id, cast(time AS DATE) AS day, date_trunc('hour',time) hour_start,
            count(*) AS rows, min(time) first_time, max(time) last_time,
            count(DISTINCT date_trunc('minute',time)) observed_minutes,
            count(*) FILTER (WHERE spl_a IS NULL) null_spl_rows
        FROM loudness WHERE {window_sql} AND sensor_id = ?
        GROUP BY 1,2,3 ORDER BY 1,2,3
    """, [sid])
    med = query('common_SPL_medians_'+sid, f"""
        SELECT sensor_id, strftime('%H',time) AS hour, count(*) AS rows,
               median(spl_a) median_spla
        FROM loudness WHERE {window_sql} AND sensor_id = ?
        GROUP BY 1,2 ORDER BY 1,2
    """, [sid])
    check(sid+' median/coverage row reconciliation', med.rows.sum() == cov.rows.sum())
    spl_coverage_parts.append(cov)
    spl_median_parts.append(med)
    print(f'SPL {index:02d}/37: {sid}; {int(cov.rows.sum()):,} raw records', flush=True)
spl_hourly = pd.concat(spl_coverage_parts, ignore_index=True)
common_spl_hourly = pd.concat(spl_median_parts, ignore_index=True)
spl_hourly.to_csv(OUT/'common_spl_hourly_coverage.csv', index=False)
common_spl_hourly.to_csv(OUT/'common_spl_hourly_medians.csv', index=False)
check('24 valid SPL clock hours per common-period sensor',
      len(common_spl_hourly)==37*24 and common_spl_hourly.groupby('sensor_id').hour.nunique().eq(24).all() and
      np.isfinite(common_spl_hourly.median_spla).all() and common_spl_hourly.median_spla.gt(0).all())

In [ ]:
coverage_parts, period_counts = [], []
for dataset, frame, old_name in [('spl',spl_hourly,'spl_raw'),('sed',sed_hourly,'sed_scores')]:
    days = frame.groupby(['sensor_id','day'],as_index=False).agg(
        rows=('rows','sum'), first_time=('first_time','min'), last_time=('last_time','max'),
        observed_minutes=('observed_minutes','sum'), observed_hours=('hour_start','size'))
    expected = old_daily[(old_daily.dataset==old_name)&old_daily.sensor_id.isin(sensors)&
                         old_daily.date.ge(START)&old_daily.date.lt(STOP)].rename(columns={'date':'day'})
    merged = days.merge(expected,on=['sensor_id','day'],suffixes=('_fresh','_audit'),how='outer',indicator=True)
    check(dataset+' exact audited sensor/day set', len(merged)==37*11 and merged['_merge'].eq('both').all())
    check(dataset+' exact audited daily row counts', merged.rows_fresh.eq(merged.rows_audit).all())
    for column in ['first_time','last_time']:
        check(dataset+' audited '+column, pd.to_datetime(merged[column+'_fresh']).eq(pd.to_datetime(merged[column+'_audit'])).all())
    check(dataset+' every sensor observed all 11 dates', days.groupby('sensor_id').day.nunique().eq(11).all())
    if dataset=='sed':
        check('SED audited observed minutes/hours',
              merged.observed_minutes_fresh.eq(merged.observed_minutes_audit).all() and
              merged.observed_hours_fresh.eq(merged.observed_hours_audit).all())
    days['dataset']=dataset
    days['observed_minute_occupancy_pct']=100*days.observed_minutes/1440
    days['below_90pct_minute_occupancy']=days.observed_minutes < 0.9*1440
    coverage_parts.append(days)
    for sid,g in days.groupby('sensor_id'):
        period_counts.append(dict(dataset=dataset,sensor_id=sid,rows=int(g.rows.sum()),
            observed_days=len(g),observed_hours=int(g.observed_hours.sum()),
            observed_minutes=int(g.observed_minutes.sum()),calendar_minutes=11*1440,
            observed_minute_occupancy_pct=100*g.observed_minutes.sum()/(11*1440)))
daily_coverage = pd.concat(coverage_parts,ignore_index=True)
daily_coverage.to_csv(OUT/'common_daily_coverage.csv',index=False)
pd.DataFrame(period_counts).to_csv(OUT/'common_period_coverage.csv',index=False)
check('sensor 001 short SED day documented', int(sed_hourly.loc[
    (sed_hourly.sensor_id=='citizen-001') & (sed_hourly.day==pd.Timestamp('2025-06-07')),
    'observed_minutes'].sum()) == 120)
display(daily_coverage[daily_coverage.below_90pct_minute_occupancy])
print('Counts and timestamp bounds reconcile with the coverage audit.')

## 3. Magnitudes and location rankings

Differences are **common period minus full period**. Ranks use descending
values (rank 1 = largest), average ranks for ties, and the identical 37-sensor
cohort. Summary medians give each sensor equal weight. Correlations use Spearman rank coefficients.

In [ ]:
common_sed = []
for sid,g in sed_hourly.groupby('sensor_id'):
    for label in CLASSES:
        valid,hits,rows = int(g[label+'_valid'].sum()),int(g[label+'_hits'].sum()),int(g.rows.sum())
        check(sid+' '+label+' valid common denominator', valid>0 and 0<=hits<=valid<=rows)
        common_sed.append(dict(sensor_id=sid,indicator=label,rows=rows,valid_scores=valid,
            null_scores=rows-valid,hits=hits,value=100*hits/valid))
common_sed = pd.DataFrame(common_sed)
common_sed.to_csv(OUT/'common_period_sed_counts.csv',index=False)
sed_results = full_sed[['sensor_id','indicator','value']].merge(
    common_sed[['sensor_id','indicator','value']],on=['sensor_id','indicator'],suffixes=('_full','_common'))
sed_results['unit']='percentage points'
full_spl = full_hourly.groupby('sensor_id').median_spla.median().map(lambda x:10*np.log10(x))
new_spl = common_spl_hourly.groupby('sensor_id').median_spla.median().map(lambda x:10*np.log10(x))
spl_results = pd.DataFrame({'value_full':full_spl,'value_common':new_spl}).reset_index()
spl_results['indicator']='spl_overview'
spl_results['unit']='dB'
results = pd.concat([sed_results,spl_results],ignore_index=True)
results['difference_common_minus_full']=results.value_common-results.value_full
results['absolute_difference']=results.difference_common_minus_full.abs()
for period in ['full','common']:
    results['rank_'+period]=results.groupby('indicator')['value_'+period].rank(ascending=False,method='average')
results['rank_change']=results.rank_common-results.rank_full
results=results.sort_values(['indicator','sensor_id']).reset_index(drop=True)
check('111 finite per-sensor results',len(results)==111 and np.isfinite(results[['value_full','value_common']]).all().all())
check('SED percentage bounds',results.loc[results.indicator.isin(CLASSES),['value_full','value_common']].ge(0).all().all() and results.loc[results.indicator.isin(CLASSES),['value_full','value_common']].le(100).all().all())
summary_rows=[]
for indicator,g in results.groupby('indicator',sort=False):
    rho=float(spearmanr(g.value_full,g.value_common).statistic)
    check(indicator+' rank correlation arithmetic',np.isclose(rho,g.rank_full.corr(g.rank_common)))
    summary_rows.append(dict(indicator=indicator,sensors=len(g),unit=g.unit.iloc[0],spearman_rho=rho,
        median_signed_change=g.difference_common_minus_full.median(),
        median_absolute_change=g.absolute_difference.median(),max_absolute_change=g.absolute_difference.max()))
summary=pd.DataFrame(summary_rows)
largest=results.sort_values('absolute_difference',ascending=False).groupby('indicator',sort=False).head(3)
results.to_csv(OUT/'per_sensor_comparison.csv',index=False)
summary.to_csv(OUT/'summary.csv',index=False)
largest.to_csv(OUT/'largest_changes.csv',index=False)
display(summary)
display(largest[['indicator','sensor_id','value_full','value_common','difference_common_minus_full','rank_full','rank_common']])

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','svg.fonttype':'none'})
fig,axes=plt.subplots(1,3,figsize=(14,5.4))
labels={'human':('Human sounds','Presence ratio (%)','#c06925'),
        'bird':('Bird sounds','Presence ratio (%)','#34816a'),
        'spl_overview':('SPL overview indicator','Sound level (dB(A))','#2676a5')}
for ax,indicator in zip(axes,labels):
    g=results[results.indicator==indicator]
    title,unit,color=labels[indicator]
    lo=float(g[['value_full','value_common']].min().min())
    hi=float(g[['value_full','value_common']].max().max())
    padding=max((hi-lo)*0.08,0.5)
    limits=(max(0,lo-padding),hi+padding) if indicator!='spl_overview' else (lo-padding,hi+padding)
    ax.plot(limits,limits,color='#858d94',ls='--',lw=1,zorder=1)
    ax.scatter(g.value_full,g.value_common,c=color,s=34,alpha=.8,zorder=2)
    for row in g.nlargest(3,'absolute_difference').itertuples():
        ax.annotate(row.sensor_id.split('-')[1],(row.value_full,row.value_common),
                    xytext=(5,5),textcoords='offset points',fontsize=8,color='#374151')
    ax.set(xlim=limits,ylim=limits,xlabel='Full available period\n'+unit,
           ylabel='30 May–9 June 2025\n'+unit)
    ax.set_aspect('equal',adjustable='box')
    r=summary.set_index('indicator').loc[indicator]
    ax.set_title(f'{title}\nSpearman rho = {r.spearman_rho:.3f}',fontsize=12)
    ax.grid(alpha=.18)
    ax.spines[['top','right']].set_visible(False)
fig.suptitle('Sensitivity to the observation period',fontsize=17,fontweight='bold',y=.98)
fig.subplots_adjust(left=.07,right=.985,bottom=.26,top=.81,wspace=.34)
fig.text(.07,.09,'Same 37 sensors in both periods; sensor 019 excluded from this check only. Dashed line: no change.\nLabels identify the three largest absolute changes per indicator. Partial days are retained; this is not a seasonal correction.',fontsize=10,color='#4b5563')
fig.savefig(OUT/'period_sensitivity.png',dpi=200,facecolor='white')
fig.savefig(OUT/'period_sensitivity.svg',facecolor='white')
plt.show()

In [ ]:
for relative,digest in source_hashes.items():
    check('input unchanged: '+relative,sha(ROOT/relative)==digest)
end_stat=database_path().stat()
check('database file unchanged',db_fingerprint==(end_stat.st_size,end_stat.st_mtime_ns))
for name,frame in [('summary.csv',summary),('per_sensor_comparison.csv',results)]:
    loaded=pd.read_csv(OUT/name)
    check(name+' row count roundtrip',len(loaded)==len(frame))
    for column in frame.select_dtypes('number'):
        check(name+' numeric roundtrip '+column,np.allclose(loaded[column],frame[column],rtol=1e-12,atol=1e-12))
manifest=dict(started_utc=started,completed_utc=datetime.now(timezone.utc).isoformat(),
    python=sys.executable,duckdb_version=duckdb.__version__,scipy_version=scipy.__version__,
    database_read_only=True,window_start_inclusive=START,window_stop_exclusive=STOP,
    sensors=sensors,excluded_sensor='citizen-019',checks=checks,queries=queries,view_sql=views,
    source_sha256=source_hashes,
    notebook_code_sha256=hashlib.sha256('\n'.join(''.join(c['source']) for c in json.loads(
        (ROOT/'analysis_10_observation_period_sensitivity.ipynb').read_text(encoding='utf-8'))['cells']).encode('utf-8')).hexdigest(),
    baseline_provenance={'SED':'Full-period source-lake counts (strict >0.5)',
       'SPL':'Reused materialized loudness_per_hour; no date-level historical lineage in that table'},
    fresh_computation='Full- and common-period SED plus common-period SPL; no inference or preprocessing')
(OUT/'checks.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
db.close()
print(f'All {len(checks)} checks passed. Outputs:',OUT)